In [1]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv(dotenv_path='.env')

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print(OPENAI_API_KEY[:5])


gsk_R


In [2]:

import langchain
print(langchain.__version__)

0.3.27


In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

from pprint import pprint

# Step 1: 사용자가 입력한 장르에 따라 영화 추천
prompt1 = ChatPromptTemplate([
  ("system", "당신은 여행 전문가입니다."),
  ("user", "{place} 의 대표적인 관광 명소 한 곳을 추천해 이름만을 알려주세요.")])

# Step 2: 추천된 관광 명소의 정보를 요약
prompt2 = ChatPromptTemplate([
  ("system","관광 명소의 정보를 요약해서 알려주세요."),
  ("user","{place} 추천한 관광 명소의 이름을 먼저 알려주시고, 줄을 바꾸어서 관광 명소의 역사, 특징, 방문 팁을 순서대로 알려 주세요.")])

# OpenAI 모델 사용
llm = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    base_url="https://api.groq.com/openai/v1",  # Groq API 엔드포인트
    #model="meta-llama/llama-4-scout-17b-16e-instruct",  # Spring AI와 동일한 모델
    model="moonshotai/kimi-k2-instruct-0905",
    temperature=0.7
)

# 체인 1: 장소 추천 (입력: 도시나 국가 → 출력: 관광 명소)
chain1 = prompt1 | llm | StrOutputParser()

# 체인 2: 관광 명소 정보 요약 (입력: 관광 명소 이름 → 출력: 정보 요약)
chain2 = (
    {"place": chain1}  # chain1의 출력을 place 변수로 전달
    | prompt2
    | llm
    | StrOutputParser()
)

place = input("관광 명소를 추천받고 싶은 도시나 국가를 입력하세요 (예: 파리, 일본): ")

response = chain1.invoke({"place": place})
print("\n🔹 1단계 결과: ")
pprint(response)
response = chain2.invoke({"place": place})
print(f"\n🔹 2단계 결과: \n")
pprint(response)


🔹 1단계 결과: 
'콜로세움'

🔹 2단계 결과: 

('콜로세움\n'
 '\n'
 '역사\n'
 '- 72년 황제 베스파시아누스 시대에 착공, 80년 티투스 개장\n'
 '- 로마 제국의 원형경기장으로 5만 명 수용, 광투·야수 사냥·해상 전투 재현 등 공연\n'
 '- 5세기까지 사용 후 방치·지진으로 파손, 18세기 중반까지 석재 채석장 역할\n'
 '\n'
 '특징\n'
 '- 타원형(직경 189×156 m, 높이 48 m) 4층 구조, 총 80개 출입구\n'
 '- 지하 하우파지(배경·가설설비 공간)와 복도식 좌석이 구조공학적 특기; 촉구 없이 쌓은 아치·돔 기술\n'
 '- 밤이면 외벽 외부 조명이 적용돼 로마의 상징적 야경 포인트\n'
 '\n'
 '방문 팁\n'
 '- 예약 없음 시 대기 1~2 h, 공식 웹사이트·통합티켓(Pass)로 시간 지정 예약 후 바로 입장\n'
 '- 로마 유적 통합티켓(콜로세움·로마 포룸·팔라티노 언덕 24/48 h) 구매 시 재입장 가능\n'
 '- 오전 8:30 개장 직방이나 1시간 전 야간투어(4~10월)가 사진·혼잡 최소화\n'
 '- 휴대용 물·모자·선크림 필수, 바닥 불규칙·계단 많음: 편한 신발 착용\n'
 '- 근처 지하철 Line B/B「Colosseo」역 바로 앞, 반경 300 m 내 카페·식당은 가격이 높음')
